# Parallel multi-site PV experiment

This notebook reads the isolated `pv_multisite_v1` data and backtest artefacts. It does not modify the single-site `1a` experiment. Build and rerun from the project root with:

```bash
energy build-multisite-pv-data --project-root . --year 2024
energy run-pv-multisite-experiment --project-root . --year 2024
```


In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "working" else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / "data/features/pv_multisite_v1/day_ahead_pv_multisite_2024.parquet"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts/experiments/pv_multisite_v1/2024"
SITE_METADATA_PATH = PROJECT_ROOT / "data/metadata/pv_multisite_v1_sites.json"

multisite = pd.read_parquet(DATA_PATH)
site_metadata = json.loads(SITE_METADATA_PATH.read_text())
print("Rows by site:")
display(multisite.groupby("site_id").size().rename("rows").to_frame())
site_metadata


In [ ]:
fold_metrics = pd.read_csv(ARTIFACT_DIR / "fold_metrics.csv")
summary = (
    fold_metrics
    .groupby(["site_id", "variant"])[["mae_w", "rmse_w"]]
    .agg(["mean", "std"])
    .round(2)
)
summary


In [ ]:
# Compare the original target site with the pooled models.
one_a = (
    fold_metrics.loc[fold_metrics["site_id"].eq("1a")]
    .pivot(index="fold", columns="variant", values="mae_w")
)
one_a.assign(
    direct_gain_w=one_a["day_ahead_baseline"] - one_a["direct_residual_catboost"],
    mlp_gain_w=one_a["day_ahead_baseline"] - one_a["mimo_mlp"],
    lstm_gain_w=one_a["day_ahead_baseline"] - one_a["mimo_lstm"],
)
